In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LightSource
from ipywidgets import interact, FloatSlider

# ⚡ 替换字体优先级，优先使用支持角标和中文的微软雅黑（Microsoft YaHei）
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

def helfrich_analytical_shape(v, c0):
    r_norm = np.linspace(0, 0.999, 100)
    phi = np.linspace(0, 2 * np.pi, 100)
    R_GRID, PHI = np.meshgrid(r_norm, phi)

    alpha = (1.0 - v) * 2.8
    c1 = 0.207 + 0.1 * alpha
    c2 = 2.0 * alpha
    c3 = -1.1 * alpha

    Z_half = 0.5 * np.sqrt(np.maximum(0, 1 - R_GRID**2)) * (c1 + c2 * R_GRID**2 + c3 * R_GRID**4)
    asymmetry = c0 * 0.25 * (1.0 - R_GRID**2)
    
    Z_upper = Z_half + asymmetry
    Z_lower = -Z_half + asymmetry

    R_full = np.vstack([R_GRID[::-1], R_GRID])
    PHI_full = np.vstack([PHI[::-1], PHI])
    Z_full = np.vstack([Z_lower[::-1], Z_upper])

    X = R_full * np.cos(PHI_full)
    Y = R_full * np.sin(PHI_full)
    Z = Z_full

    r_2d_1d = r_norm
    z_up_1d = Z_upper[0, :]
    z_low_1d = Z_lower[0, :]

    return X, Y, Z, r_2d_1d, z_up_1d, z_low_1d

def plot_helfrich_model(v=0.65, c0=0.0):
    X, Y, Z, r_2d, z_up, z_low = helfrich_analytical_shape(v, c0)

    fig = plt.figure(figsize=(13, 6), facecolor='#f8f9fa')

    # 1. 3D 高质感曲面
    ax1 = fig.add_subplot(121, projection='3d', facecolor='#f8f9fa')
    ls = LightSource(azdeg=135, altdeg=45)
    rgb = ls.shade(Z, cmap=plt.cm.coolwarm, vert_exag=0.1, blend_mode='soft')

    surf = ax1.plot_surface(X, Y, Z, rstride=1, cstride=1, facecolors=rgb,
                            linewidth=0, antialiased=True, shade=False)
    
    ax1.set_xlim([-1.1, 1.1])
    ax1.set_ylim([-1.1, 1.1])
    ax1.set_zlim([-1.1, 1.1])
    ax1.view_init(elev=25, azim=50)
    ax1.axis('off')
    ax1.set_title(f"Helfrich 膜 3D 构型\n(体积系数 v={v:.2f}, 自发曲率 c0={c0:.2f})", fontsize=12, fontweight='bold', pad=10)

    # 2. 2D 精确剖面图
    ax2 = fig.add_subplot(122, facecolor='#ffffff')
    r_contour = np.concatenate([-r_2d[::-1], r_2d])
    z_up_contour = np.concatenate([z_up[::-1], z_up])
    z_low_contour = np.concatenate([z_low[::-1], z_low])

    ax2.plot(r_contour, z_up_contour, color='#d90429', lw=2.5, label='上层脂质膜')
    ax2.plot(r_contour, z_low_contour, color='#023e8a', lw=2.5, label='下层脂质膜')
    
    ax2.fill_between(r_contour.ravel(), z_low_contour.ravel(), z_up_contour.ravel(), 
                     color='#ffb703', alpha=0.25, label='细胞内体积 (V)')

    ax2.set_xlim([-1.2, 1.2])
    ax2.set_ylim([-0.8, 0.8])
    ax2.set_aspect('equal')
    # 使用 LaTeX 数学公式渲染 R_0，彻底避免字体兼容警告
    ax2.set_xlabel("归一化半径 (r / $R_0$)", fontsize=10)
    ax2.set_ylabel("轴向高度 (z / $R_0$)", fontsize=10)
    ax2.set_title("Helfrich 形状子午截面 (Cross-section)", fontsize=12, fontweight='bold')
    ax2.grid(True, linestyle=':', alpha=0.6)
    ax2.legend(loc='upper right', frameon=True, facecolor='white', framealpha=0.9)

    for spine in ax2.spines.values():
        spine.set_color('#cccccc')

    plt.tight_layout()
    plt.show()

interact(
    plot_helfrich_model, 
    v=FloatSlider(min=0.55, max=1.0, step=0.02, value=0.65, continuous_update=False, description='体积系数 (v)'),
    c0=FloatSlider(min=-1.0, max=1.0, step=0.1, value=0.0, continuous_update=False, description='自发曲率 (c0)')
);

interactive(children=(FloatSlider(value=0.65, continuous_update=False, description='体积系数 (v)', max=1.0, min=0.…